### Wegemer, SOC 180C Fall 2025
# Class 6 - Sampling and Normal Distributions

## Is school discipline biased?

First, let's load data on expulsions from a high school, Hypothetical High. 

In [ ]:
hypodata <- read.csv(url("https://raw.githubusercontent.com/cwegemer/SOC180/main/Class6-Hypotheticalschooldata.csv"))
head(hypodata)

One year at the high school, 10 students were expelled. Only two of these students were white. Upset parents accused the school of racial bias. Assume you work for a community legal organization and you need to determine if there is a valid civil rights case against the school.

![Alt text](https://raw.githubusercontent.com/cwegemer/SOC180/main/Class6-racialexpulsion.png?raw=true)

We are going to to select 10 students completely at random from the school, then see if they look different from the students that were expelled.

In [ ]:
expelled_students <- hypodata[sample(1:500, 10), ] # Selects 10 students at random

hist(expelled_students$race,
  main = "Race/Ethnicity of Expelled Students",
  xlab = "Race/Ethnicity",
  ylab = "Number of Students",
  ylim = c(0, 10),  # Set y-axis range from 0 to 10, for consistency
  xaxt = "n",  # Removes the default x-axis so we can put labels
  breaks = seq(0.5, 6.5, by = 1) # This moves the bars over by .5 so that the columns will line up with the labels better
)
axis(1, # This puts labels on the x-axis
     at = c(1,2,3,4,5,6), 
     labels = c("White", "Black", "Hispanic", "Asian", "Indigenous", "Other")
)

Run the above block of code several times. Each time, compare the results with the original set of students who were expelled. (In particular, look for differences in the number of white students in each group versus all other students of color.)

To make it easier to see if there are any patterns, we are going have the computer do this 100 times, and each time, tell us the number of students of color who would be expelled if 10 students at the school were picked completely at random.

In [ ]:
results<-c() # Create an empty vector we can use to store the results

for (i in 1:100) {
  expelled_students <- hypodata[sample(1:500, 10), ]
  # Count students who are NOT white (i.e., race != 1)
  results[i] <- sum(expelled_students$race!=1)
}

results

Now, let's calculate the average:

In [ ]:
mean(results)

If it was very uncommon to see a randomly selected group of students that resembled the expelled students, then there might be racial bias. 

We always need to think...

![Alt text](https://raw.githubusercontent.com/cwegemer/SOC180/main/Class6-Samplingdistribution.png?raw=true)

Lastly, let's look at the overall racial/ethnic distribution of the entire school.

In [ ]:
hist(hypodata$race,
     main = "Race/Ethnicity of All Students at the School",
     xlab = "Race/Ethnicity",
     ylab = "Number of Students",
     ylim = c(0, 500),  # set y-axis range from 0 to 10
     xaxt = "n",  # Removes the default x-axis so we can put labels
     breaks = seq(0.5, 6.5, by = 1) # This moves the bars over by .5 so that the columns will line up with the labels better
)
axis(1, # This puts labels on the x-axis
     at = c(1,2,3,4,5,6), 
     labels = c("White", "Black", "Hispanic", "Asian", "Indigenous", "Other")
)

## Using real data

Let's use real data to investigate the number of security guards on high school campuses.

In [ ]:
hsdata <- read.csv(url("https://raw.githubusercontent.com/cwegemer/SOC180/main/Datacasestudy2-DOECRDC2021-22.csv"))
head(hsdata)

First, we need to clean our variable. 

In [ ]:
# First, change all negative values to missing 

hsdata$security_guards <- ifelse(
  hsdata$sch_ftesecurity_gua == -3 | hsdata$sch_ftesecurity_gua == -6 | hsdata$sch_ftesecurity_gua == -9,
  NA,
  hsdata$sch_ftesecurity_gua
)

mean(hsdata$security_guards, na.rm = TRUE)

## Practice exercise 1

Let's refresh our memory about how to use this data. 

Make a histogram of the number of security guards. You will likely need to adjust the axes a few times so that you can get a feel for the distribution. 

There is a school in California called Santa Maria Joint Union High that has 13 security guards. Put a line on your histogram that shows where this school is. (You might want to review code from the last lecture or your homework to help you.)

Is this school different from the general pattern? 

## Sampling

In the vast majority of research studies, we don't have data on every single case. We can only collect data on a sample. 

Run the below code to select a random sample from all of the high schools and visualize the result.

In [ ]:
sample <- hsdata[sample(1:nrow(hsdata), 200), ]

hist(sample$security_guards,
     main = "Security Guards in One Random Sample of Schools",
     xlab = "Security Guards",
     ylab = "Number of Schools")

It is possible that we randomly pick a sample that makes us think our school is normal! Instead of just relying on the average of one sample, we need to look at many samples.

Before we start, run the below line of code.

In [ ]:
# This creates an empty vector that we will use next
sampling_means_so_far <- c()
# You can also run this code to re-set the histogram that we will create next...

Every time you run the below code, it takes a random sample of 200 schools, calculates the average, and adds the sample to the histogram.

Run the below code many times to see what happens when we repeatedly select random samples of schools.

In [ ]:
sample_size <- 200
new_sample <- sample(hsdata$security_guards, sample_size, replace = TRUE)
new_mean <- mean(new_sample, na.rm = TRUE)

sampling_means_so_far <- c(sampling_means_so_far, new_mean)

# Step 5: Plot sampling distribution
hist(sampling_means_so_far,
     breaks = 20,
     main = paste("Sampling Distribution (", length(sampling_means_so_far), " samples)", sep = ""),
     xlim = c(0, 2),
     xlab = "Average number of security guards in each sample",
     ylab = "Number of samples")

This is amazing!

## Inferring the population from one random sample

In LA Unified School District, there are 1,098 schools. On average, schools in LAUSD have 0.4 security guards per school. 

Let's say we are researchers and we want to see if LAUSD is different than other samples of schools in the country. We don't have enough resources to survey 1,098 schools though, we can only survey 500. 

In [ ]:
set.seed(999) # This code makes sure that we all see the exact same randomly generated group
sample_size <- 500

# Randomly select 500 schools
sample_study <- sample(hsdata$security_guards, sample_size, replace = TRUE)


Next, we are going to calculate the mean and standard deviation of our random sample. Using this information, we will be able to estimate the sampling distribution.

In [ ]:
sample_mean <- mean(sample_study, na.rm = TRUE)
sample_sd <- sd(sample_study, na.rm = TRUE)
n <- sum(!is.na(sample_study))
se <- sample_sd / sqrt(n) # We'll talk about what this is next class

Now, visualize the approximate sampling distribution.

In [ ]:
# The below line just picks a bunch of x values that we will use for plotting
x_vals <- seq(sample_mean - 4*se, sample_mean + 4*se, length.out = 1000)

plot(x_vals, 
     dnorm(x_vals, mean = sample_mean, sd = se), # y values of the normal distribution 
     xlim = c(0, 2),
     main = "Estimated sampling distribution for all high schools in the US",
     xlab = "Average security guards per school",
     ylab = "Number of schools")
abline(v = 0.4, col = "red")

## Reflection questions

How does this compare to the sampling distribution that we created by manually calculating the means of lots and lots of random samples?

Do you think LAUSD is different than high schools across the US? Why or why not? About what proportion of high schools do you think would have values lower than LAUSD?